# 2) EE-1.2 Sources, superposition and Thevenin — Exercises

**Goals:** real sources under load, nodal analysis with fixed voltages, superposition as a tool, Thevenin equivalents and grid strength, and a first distribution-factor matrix — the idea behind PTDFs in Module 04.

Do tasks 4 and 5 on paper first (series/parallel by hand), then check them in code. Units: V, A, Ω, W.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import math

import numpy as np
import pandas as pd

LECTURE_BRANCHES = [(1, 2, 2.0), (2, 3, 3.0), (2, 0, 10.0), (3, 0, 20.0)]   # (from, to, ohm)


def full_conductance(branches, n_nodes):
    """Conductance matrix (S) including the reference node as an ordinary node (from the lecture)."""
    G = np.zeros((n_nodes, n_nodes))
    for a, b, r in branches:
        g = 1.0 / r
        G[a, a] += g; G[b, b] += g; G[a, b] -= g; G[b, a] -= g
    return G

### Warm-ups

**1. A real source under load**

A source has open-circuit voltage `E` and internal resistance `R_int`. Write `terminal_voltage` and `source_efficiency` — the share of the source's internal power E·I that reaches the terminals.

In [2]:
def terminal_voltage(E: float, R_int: float, I: float) -> float:
    """Terminal voltage (V) of a source delivering current I (A)."""
    ### BEGIN SOLUTION
    return E - I * R_int
    ### END SOLUTION


def source_efficiency(E: float, R_int: float, I: float) -> float:
    """Delivered power / internal power E*I."""
    ### BEGIN SOLUTION
    return terminal_voltage(E, R_int, I) * I / (E * I)
    ### END SOLUTION

In [3]:
assert math.isclose(terminal_voltage(100, 0.5, 20), 90.0)
assert math.isclose(source_efficiency(100, 0.5, 20), 0.9), "10 % lost inside the source"
assert source_efficiency(100, 0.5, 40) < source_efficiency(100, 0.5, 20), "more current, lower efficiency"

**2. Nodal analysis with fixed voltages**

Write `solve_fixed(branches, n_nodes, injections_A, fixed_V)`: partition **G** into known and unknown node voltages, solve G_UU · V_U = I_U − G_UK · V_K, and return all node voltages and the net injections G · V. Test it on EE-1.1's worked example (only the reference fixed) and on the lecture network.

In [4]:
def solve_fixed(branches, n_nodes: int, injections_A, fixed_V: dict) -> tuple[np.ndarray, np.ndarray]:
    """Node voltages (V) and net injections (A); fixed_V maps node -> voltage."""
    ### BEGIN SOLUTION
    G = full_conductance(branches, n_nodes)
    K = sorted(fixed_V)
    U = [i for i in range(n_nodes) if i not in fixed_V]
    V = np.zeros(n_nodes)
    V[K] = [fixed_V[k] for k in K]
    I = np.asarray(injections_A, dtype=float)
    V[U] = np.linalg.solve(G[np.ix_(U, U)], I[U] - G[np.ix_(U, K)] @ V[K])
    return V, G @ V
    ### END SOLUTION

In [5]:
V, _ = solve_fixed([(1, 0, 4.0), (1, 2, 2.0), (2, 0, 4.0)], 3, [0, 3.0, 0], {0: 0.0})
assert np.allclose(V, [0, 7.2, 4.8]), "EE-1.1 worked example"

V, inj = solve_fixed(LECTURE_BRANCHES, 4, [0, 0, 0, 2.0], {0: 0.0, 1: 100.0})
assert math.isclose(V[3], 75.1351, rel_tol=1e-5)
assert np.allclose(inj[[2, 3]], [0.0, 2.0]), "free nodes inject exactly what was specified"
assert math.isclose(inj.sum(), 0.0, abs_tol=1e-9), "KCL for the whole network: injections sum to zero"

### Core

**3. Superposition as a tool**

Write `superposition_parts`: return a dict with one voltage vector per **source** — one for each fixed node with a non-zero voltage, one for each non-zero current injection — each computed with all other sources switched off (voltage sources at 0 V, injections at 0 A; the fixed nodes stay fixed). The parts must add up to the full solution.

In [6]:
def superposition_parts(branches, n_nodes: int, injections_A, fixed_V: dict) -> dict:
    """Node voltages caused by each source acting alone, keyed by source name."""
    ### BEGIN SOLUTION
    parts = {}
    off = {k: 0.0 for k in fixed_V}
    for k, v in fixed_V.items():
        if v != 0:
            parts[f"voltage source at node {k}"] = solve_fixed(branches, n_nodes, np.zeros(n_nodes), {**off, k: v})[0]
    for k, i in enumerate(injections_A):
        if i != 0:
            one = np.zeros(n_nodes); one[k] = i
            parts[f"injection at node {k}"] = solve_fixed(branches, n_nodes, one, off)[0]
    return parts
    ### END SOLUTION

In [7]:
inj3 = [0, 0, 0, 2.0]
parts = superposition_parts(LECTURE_BRANCHES, 4, inj3, {0: 0.0, 1: 100.0})
full, _ = solve_fixed(LECTURE_BRANCHES, 4, inj3, {0: 0.0, 1: 100.0})
assert len(parts) == 2, "one voltage source + one injection"
assert np.allclose(sum(parts.values()), full), "the parts must add up"

two_inj = [0, 0, 1.0, 2.0]                          # add a second injection at node 2
parts2 = superposition_parts(LECTURE_BRANCHES, 4, two_inj, {0: 0.0, 1: 100.0})
assert len(parts2) == 3 and np.allclose(sum(parts2.values()), solve_fixed(LECTURE_BRANCHES, 4, two_inj, {0: 0.0, 1: 100.0})[0])
pd.DataFrame(parts).round(3)

   voltage source at node 1  injection at node 3
0                     0.000                0.000
1                   100.000                0.000
2                    77.703                2.703
3                    67.568                7.568

**4. Thevenin equivalent at a terminal**

Write `thevenin(branches, n_nodes, fixed_V, terminal)` returning `(V_th, R_th)` from an open-circuit and a short-circuit experiment at `terminal` (against the reference node 0). Before running: compute both by hand for the lecture network **seen from node 2** (without the 10 Ω load at node 2).

In [8]:
def thevenin(branches, n_nodes: int, fixed_V: dict, terminal: int) -> tuple[float, float]:
    """Thevenin voltage (V) and resistance (ohm) seen between `terminal` and node 0."""
    ### BEGIN SOLUTION
    V_open, _ = solve_fixed(branches, n_nodes, np.zeros(n_nodes), fixed_V)
    V_th = V_open[terminal]
    _, inj = solve_fixed(branches, n_nodes, np.zeros(n_nodes), {**fixed_V, terminal: 0.0})
    I_sc = -inj[terminal]
    return float(V_th), float(V_th / I_sc)
    ### END SOLUTION

In [9]:
seen_from_3 = [(1, 2, 2.0), (2, 3, 3.0), (2, 0, 10.0)]
V_th3, R_th3 = thevenin(seen_from_3, 4, {0: 0.0, 1: 100.0}, 3)
assert math.isclose(V_th3, 100 * 10 / 12) and math.isclose(R_th3, 3 + 2 * 10 / 12), "lecture values"

seen_from_2 = [(1, 2, 2.0), (2, 3, 3.0), (3, 0, 20.0)]
V_th2, R_th2 = thevenin(seen_from_2, 4, {0: 0.0, 1: 100.0}, 2)
assert math.isclose(V_th2, 100 * 23 / 25), "voltage divider: 2 ohm against 3 + 20 ohm"
assert math.isclose(R_th2, 2 * 23 / 25), "2 ohm in parallel with 3 + 20 ohm"

for R_load in [1.0, 10.0, 100.0]:                    # the equivalent must behave like the network
    V_full, _ = solve_fixed(seen_from_2 + [(2, 0, R_load)], 4, np.zeros(4), {0: 0.0, 1: 100.0})
    assert math.isclose(V_full[2] / R_load, V_th2 / (R_th2 + R_load))
print(f"node 2: V_th = {V_th2:.2f} V, R_th = {R_th2:.3f} ohm | node 3: V_th = {V_th3:.2f} V, R_th = {R_th3:.3f} ohm")

node 2: V_th = 92.00 V, R_th = 1.840 ohm | node 3: V_th = 83.33 V, R_th = 4.667 ohm


**5. Interpretation**

A 5 A load (an electric-vehicle charging hub, say) wants to connect at node 2 or at node 3. Using your Thevenin results, estimate by how many volts the voltage at each point drops when the hub switches on, and recommend a connection point. Why do network operators publish short-circuit levels for exactly this purpose?

<!-- BEGIN SOLUTION -->
The voltage change is ΔV ≈ R_th · ΔI. At node 2, R_th ≈ 1.84 Ω, so a 5 A load lowers the voltage by about 9.2 V; at node 3, R_th ≈ 4.67 Ω, so about 23.3 V — two and a half times more. Node 2 is electrically closer to the source (lower R_th, higher short-circuit current), so it is the **stronger** connection point and the better choice: the hub disturbs neighbours' voltages less and keeps its own voltage within limits more easily. Operators publish short-circuit levels (short-circuit current or power at each substation) because they summarise R_th in one number: a developer can immediately estimate voltage changes, check whether a large load or generator fits, and see where reinforcement would be needed. Weak points (high R_th) suffer larger voltage swings, flicker and, for inverter-based generation, control stability problems.
<!-- END SOLUTION -->

### Challenge

**6. A first distribution-factor matrix**

In a meshed network with only the reference node fixed (0 V), inject **1 A** at one node (it returns through the reference). The current in every branch is a fixed fraction of that ampere — a **distribution factor**. Write `distribution_factors(branches, n_nodes)` returning a DataFrame (rows: branches as `"a-b"`, columns: injection nodes 1 … n−1) of branch currents (A, positive from a to b) per 1 A injected. Then predict the branch currents for a combined injection by superposition — one matrix–vector product — and check against a direct solve.

In [10]:
MESH = [(1, 2, 1.0), (2, 3, 2.0), (1, 3, 2.0), (1, 0, 1.0), (3, 0, 4.0)]


def distribution_factors(branches, n_nodes: int) -> pd.DataFrame:
    """Branch currents (A, a->b) per 1 A injected at each non-reference node, returning via node 0."""
    ### BEGIN SOLUTION
    cols = {}
    for k in range(1, n_nodes):
        inj = np.zeros(n_nodes); inj[k] = 1.0
        V, _ = solve_fixed(branches, n_nodes, inj, {0: 0.0})
        cols[k] = [(V[a] - V[b]) / r for a, b, r in branches]
    return pd.DataFrame(cols, index=[f"{a}-{b}" for a, b, _ in branches])
    ### END SOLUTION

In [11]:
DF = distribution_factors(MESH, 4)
assert DF.shape == (5, 3)
into_ref = DF.loc["1-0"] + DF.loc["3-0"]
assert np.allclose(into_ref, 1.0), "every injected ampere returns through the reference (KCL)"

injection = np.array([2.0, -1.0, 3.0])               # A at nodes 1, 2, 3
predicted = DF.to_numpy() @ injection
V, _ = solve_fixed(MESH, 4, np.concatenate([[0.0], injection]), {0: 0.0})
direct = np.array([(V[a] - V[b]) / r for a, b, r in MESH])
assert np.allclose(predicted, direct), "superposition: one matrix product gives every branch current"
DF.round(4)

          1       2       3
1-2  0.0645 -0.7097 -0.2581
2-3  0.0645  0.2903 -0.2581
1-3  0.0968 -0.0645 -0.3871
1-0  0.8387  0.7742  0.6452
3-0  0.1613  0.2258  0.3548

**7. Interpretation**

Grid operators use distribution factors (PTDFs) to decide in milliseconds whether a trade between two areas would overload a line. Explain why superposition makes that possible, and give two reasons why distribution factors for **real AC power** flows are only approximations (think of M1.3 and of this lesson's power example).

<!-- BEGIN SOLUTION -->
Because the network is linear, the flow on every line is a weighted sum of all injections, with weights (the distribution factors) that depend only on the network, not on the injections. The operator computes the matrix once; checking any trade is then one multiplication per line — no network solve — so thousands of trades or contingencies can be screened almost instantly. For real AC power flows this is an approximation: (1) the AC power flow equations are **nonlinear** — power depends on products of voltages and on angles (M1.3: constant-power loads already made the DC case nonlinear), so PTDFs come from a **linearised** (DC) power flow that assumes flat voltages, small angle differences and negligible resistance; (2) **losses** are quadratic in current, as this lesson showed, so they cannot be represented by linear factors at all; further, reactive power and voltage limits are ignored, and the factors change whenever the topology changes (a line trips) — which is why operators recompute them and add margins. Module 04 derives DC PTDFs and tests where they break.
<!-- END SOLUTION -->